In [1]:
# Imports
import pandas as pd
import numpy as np
import torch
import tempfile
import warnings
import sys
import time
import uuid
import os

sys.path.append("..")

from scheduler.minio.create import MinioConnection  # Ensure MinioConnection is available
from gat.utils.save import Saver  # Import the Saver class
from nets.utils.load import Loader  # Import the Loader class
from nets.data import CustomDataset
from nets.model import CategoricalInput, NumericalInput, ImageInput, TextInput

warnings.filterwarnings("ignore", category=FutureWarning)

# Map model types to their respective classes
model_class_map = {
    "CATEGORY": CategoricalInput,
    "NUMBER": NumericalInput,
    "IMAGE": ImageInput,
    "TEXT": TextInput
}

In [2]:
def train(data):
    """
    Train a model based on the provided data.

    Args:
        data (dict): A dictionary containing the following keys:
            - 'dtype' (str): The type of model to train (e.g., "CATEGORY", "NUMBER", "IMAGE", "TEXT").
            - 'd_params' (dict): Parameters for the dataset and training process.
            - 'model_path' (str, optional): Path to save the trained model. If not provided, a default path is used.

    Returns:
        dict: A response dictionary containing a success message, the model path, and an empty traceback.
    """
    # Extract model type and dataset parameters
    dtype = data.get('dtype')
    d_params = data['d_params']

    # Generate a unique filename for the model
    unique_filename = f"{dtype}_{int(time.time())}_{uuid.uuid4().hex}.pt"
    model_path = f"/mnt/models/{prefix}/{unique_filename}"  # Define the full path to the model

    # Initialize the dataset
    dataset = CustomDataset(d_params, dtype)
    dataset.reset_data(training=True)  # Reset the dataset for training

    # Initialize the model
    model_class = model_class_map[dtype]
    model = model_class(dataset, d_params)
    optimizer = torch.optim.SGD(model.parameters(), lr=d_params['h_params']['learning_rate'])
    model.optimizer = optimizer

    # Train the model
    model.train_model(dataset.dataloader, d_params['h_params']['epochs'], optimizer)

    # Save the trained model using the Saver class
    minio_connection = MinioConnection()  # Initialize your MinIO connection
    
    saver = Saver()
    saver.save_model(
        params=model.state_dict(),
        combined_metrics={},  # Example metrics
        minio_connection=minio_connection,
        unique_filename=unique_filename,
        prefix=prefix,
    )

    # Return a success response with the model path
    return model_path

In [3]:
def predict(data):
    """
    Predict using a trained model.

    This function loads a trained model, preprocesses the input data, and performs predictions.

    Args:
        data (dict): A dictionary containing the following keys:
            - 'model_path' (str): Path to the model file.
            - 'd_params' (dict): Dataset parameters.
            - 'new_data' (dict, optional): New data for prediction. Defaults to 'd_params.x'.
    """
    # Extract the model path and new data for prediction
    model_path = data.get('model_path')
    d_params = data.get('d_params')
    dtype = data.get('dtype')
    new_data = data.get('new_data', d_params['x'])  # Use 'new_data' if provided, otherwise use 'd_params.x'

    # Initialize the Loader
    loader = Loader()

    # Load the trained model using the Loader
    model_class = model_class_map[dtype]
    model, _, _ = loader.load_model(
        unique_filename=os.path.basename(model_path),  # Extract the filename from the model path
        model_class=model_class,
        bucket_name="models",  # Replace with the appropriate bucket name
        prefix=prefix  # Use the prefix defined in your script
        d_params=d_params
    )

    # Initialize the dataset with the provided parameters
    dataset = CustomDataset(d_params, dtype)
    dataset.reset_data(training=False)  # Reset the dataset for prediction

    # Preprocess the new data to match the model's input format
    input_tensor = preprocess_new_data(dataset, new_data)

    # Perform prediction using the trained model
    model.eval()  # Set the model to evaluation mode
    with torch.no_grad():  # Disable gradient computation for inference
        output = model.forward(input_tensor)  # Perform a forward pass
        predictions = output.tolist()  # Convert the output tensor to a list

    # Print the predictions
    print("Predictions:", predictions)

In [4]:
def explain(data):
    """
    Explain the predictions of a trained model using SHAP values.

    This function loads a trained model, preprocesses the input data, and computes SHAP values
    to explain the model's predictions.
    """
    # Extract the model path, data parameters, and new data for explanation
    model_path = data.get('model_path')
    d_params = data.get('d_params')
    new_data = data.get('new_data', d_params['x'])  # Use 'new_data' if provided, otherwise use 'd_params.x'
    dtype = data['dtype']

    # Initialize the Loader
    loader = Loader()

    # Load the trained model using the Loader
    model, _, _ = loader.load_model(
        unique_filename=os.path.basename(model_path),
        bucket_name="models",
        prefix=prefix,
        resume_training=False
    )

    # Load the SHAP explainer using the Loader
    shap_explainer = loader.load_explainer(
        model_class_name=d_params.get('model_class_name'),
        d_params=d_params
    )

    if shap_explainer is None:
        print("Error: SHAP explainer could not be loaded. Ensure it exists for the specified model.")
        return

    # Initialize the dataset with the provided parameters
    dataset = CustomDataset(d_params, dtype)
    dataset.reset_data(training=False)  # Reset the dataset for explanation

    # Preprocess the new data to match the model's input format
    input_tensor = preprocess_new_data(dataset, new_data)

    # Perform explanation using the SHAP explainer
    shap_values = shap_explainer.shap_values(input_tensor)
    expected_value = shap_explainer.expected_value

    # Convert SHAP values and expected value to a JSON-serializable format
    shap_values_serializable = [sv.tolist() for sv in shap_values]  # Convert SHAP values to lists
    expected_value_serializable = (
        expected_value.tolist() if isinstance(expected_value, np.ndarray) else expected_value
    )  # Convert expected value to a list if it's a NumPy array

    # Print the SHAP values and expected value
    print("SHAP Values:", shap_values_serializable)
    print("Expected Value:", expected_value_serializable)

In [5]:
def preprocess_new_data(dataset, new_data):
    """
    Preprocess the new data using the same preprocessing steps as the training data.

    This function prepares new data for prediction or explanation by converting it into a format
    compatible with the model's input requirements. It saves the preprocessed data to a temporary
    file and updates the dataset object.

    Args:
        dataset (CustomDataset): The dataset object containing preprocessing methods and metadata.
        new_data (dict): The new data to preprocess, structured as a dictionary where keys are
                         feature names and values are dictionaries containing feature types and values.

    Returns:
        torch.Tensor: The preprocessed input tensor ready for model inference.
    """
    # Extract values from the new_data dictionary to create a DataFrame
    new_data_values = {key: value["values"] for key, value in new_data.items()}
    new_data_df = pd.DataFrame(new_data_values)

    # Exclude target columns (output labels) from the new data
    target_columns = list(dataset.d_params["y"].keys())  # Get the target column names
    new_data_df = new_data_df.drop(columns=target_columns, errors='ignore')  # Drop target columns if they exist

    # Save the preprocessed new data to a temporary CSV file
    with tempfile.NamedTemporaryFile(delete=False, suffix='.csv') as tmp_file:
        new_data_df.to_csv(tmp_file.name, index=False)  # Save the DataFrame to the temporary file
        temp_filepath = tmp_file.name  # Store the temporary file path

    # Update the dataset object with the temporary file path and reload the data
    dataset.filepath = temp_filepath  # Set the dataset's file path to the temporary file
    dataset.update_data(training=False)  # Update the dataset for inference (not training)

    # Retrieve all input tensors from the dataset
    input_tensor, _ = dataset.get_all_tensors(training=False)  # Get input tensors for inference

    # Return the preprocessed input tensor
    return input_tensor

In [6]:
# CATEGORY Data Type
prefix = "rory/test_CATEGORICAL"
unique_filename = f"model_{int(time.time())}_{uuid.uuid4().hex}.pth"

data = {
    "dtype": "CATEGORY",
    "d_params": {
        "x": {
            "location_1": {"type": "CATEGORY", "values": ["California", "Iowa"]},
            "location_2": {"type": "CATEGORY", "values": ["Monterey County", "Osceola County"]}
        },
        "y": {
            "crop_1": {"type": "CATEGORY", "values": ["Corn", "Wheat"]}
        },
        "h_params": {"learning_rate": 0.01, "epochs": 20, "batch_size": 32},
        "file_paths": [f"/mnt/data/agtech/test_project/labels.csv"]
    }
}

data["model_path"] = train(data)
predict(data)
explain(data)

TypeError: Loader.load_model() missing 1 required positional argument: 'd_params'

In [ ]:
# NUMBER Data Type
prefix = "rory/test_NUMERICAL"
unique_filename = f"model_{int(time.time())}_{uuid.uuid4().hex}.pth"

data_number = {
    "dtype": "NUMBER",
    "model_path": "/mnt/models/NumericalInput/model.pth",
    "d_params": {
        "x": {
            "temperature": {"type": "NUMBER", "values": [72.5, 85.3]},
            "humidity": {"type": "NUMBER", "values": [45.2, 60.1]}
        },
        "y": {
            "yield": {"type": "NUMBER", "values": [200, 250]}
        },
        "h_params": {"learning_rate": 0.001, "epochs": 50, "batch_size": 16},
        "file_paths": [f"/mnt/data/agtech/test_project/labels.csv"]
    }
}

data["model_path"] = train(data)
predict(data_number)
explain(data_number)

In [ ]:
!ls /mnt/data/agtech/test_project

In [ ]:
# IMAGE Data Type
prefix = "rory/test_IMAGE"
unique_filename = f"model_{int(time.time())}_{uuid.uuid4().hex}.pth"

data_image = {
    "dtype": "IMAGE",
    "model_path": "/mnt/data/models/ImageInput/model.pth",
    "d_params": {
        "x": {
            "image_paths": {"type": "IMAGE", "values": ["/mnt/data/images/img1.jpg", "/mnt/data/images/img2.jpg"]}
        },
        "y": {
            "labels": {"type": "CATEGORY", "values": ["Healthy", "Diseased"]}
        },
        "h_params": {"learning_rate": 0.0001, "epochs": 30, "batch_size": 8},
        "file_paths": [f"/mnt/data/agtech/test_project/labels.csv"]
    }
}

data["model_path"] = train(data)
predict(data_image)
explain(data_image)

In [ ]:
# TEXT Data Type
prefix = "rory/test_TEXT"
unique_filename = f"model_{int(time.time())}_{uuid.uuid4().hex}.pth"

data_text = {
    "dtype": "TEXT",
    "model_path": "/mnt/data/models/TextInput/model.pth",
    "d_params": {
        "x": {
            "reviews": {"type": "TEXT", "values": ["The crop is healthy", "The crop is diseased"]}
        },
        "y": {
            "sentiment": {"type": "CATEGORY", "values": ["Positive", "Negative"]}
        },
        "h_params": {"learning_rate": 0.0005, "epochs": 10, "batch_size": 16},
        "file_paths": [f"/mnt/data/agtech/test_project/labels.csv"]
    }
}

data["model_path"] = train(data)
predict(data_text)
explain(data_text)